In [1]:
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
from typing import Literal
from langchain_core.tools import tool
from langgraph.graph import MessagesState, START, END, StateGraph

In [2]:
@tool
def send_email(to: str, body: str) -> str:
    """Send an email"""
    return f"Email sent to {to}"

In [3]:
# fake "agent"; this node is supposed to call an llm with tools
def agent(state: MessagesState) -> dict:
    # pretend the LLM decided to call send_email
    from langchain_core.messages import AIMessage
    msg = AIMessage(content="", tool_calls=[{
          "name": "send_email",
           "args": {"to": "team@x.com", "body": "Deploying at 5pm"},
           "id": "call_1"
    }])
    return {"messages": [msg]}   


In [4]:
def review_tool_call(state: MessagesState):
    last = state["messages"][-1]

    decision = interrupt({
        "question": "Approve this tool call?",
        "tool_calls": last.tool_calls,
    })

    if decision["type"] == "approve":
        return Command(goto="tools")
    elif decision["type"] == "reject":
        return Command(
            update={"messages": [{
                "role": "tool",
                "tool_call_id": last.tool_calls[0]["id"],
                "content": "Rejected by human."
            }]},
            goto="agent"
        )

## interrupt, Command eg

In [5]:
from langgraph.graph import START, END, MessagesState, StateGraph
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from typing import TypedDict

In [6]:
class State(TypedDict):
    name: str
    greeting: str

In [13]:
def ask_name(state: State) -> dict:
    print("Node started")
    name = interrupt("Enter name: ")
    print(f"Node resumed, {name}")
    return {"name": name}

In [15]:
def greet(state: State) -> dict:
    return {"greeting": f"Hello, {state['name']}!"}

In [16]:
builder = StateGraph(State)
builder.add_node("ask_name", ask_name)
builder.add_node("greet", greet)

builder.add_edge(START, "ask_name")
builder.add_edge("ask_name", "greet")
builder.add_edge("greet", END)

In [17]:
graph = builder.compile(checkpointer=InMemorySaver())

In [18]:
config = {"configurable": {"thread_id": "t1"}}

res1 = graph.invoke({}, config=config)
print(res1)

Node started
{'__interrupt__': [Interrupt(value='Enter name: ', id='b270a82767a86111b099ba1d600f4d33')]}


In [19]:
res2 = graph.invoke(Command(resume="Geet"), config=config)
print(res2)

Node started
Node resumed, Geet
{'name': 'Geet', 'greeting': 'Hello, Geet!'}
